# ISS Group 24 Modeling

**Before running:**
1. Open the [shared Drive folder](https://drive.google.com/drive/folders/19el_ISdRf5WoXarBsUOXjxFY9A4cPGYJ?usp=sharing) and click **"Add shortcut to Drive"** (anywhere in your Drive is fine).
2. Select a GPU runtime: *Runtime → Change runtime type → T4 GPU*.
3. Run cells **in order** top-to-bottom. Each stage cell is idempotent — re-running a completed stage skips it automatically via checkpoint detection.

In [1]:
# Enable autoreload so subsequent edits to modeling/*.py are picked up
# without restarting the kernel. Idempotent — safe to re-run.
%load_ext autoreload
%autoreload 2

import os
import sys
import subprocess
from pathlib import Path


In [2]:
SHARED_FOLDER_LINK = "https://drive.google.com/drive/folders/19el_ISdRf5WoXarBsUOXjxFY9A4cPGYJ?usp=sharing"
REPO_URL        = "https://github.com/pewpewnor/iss_group_24.git"
REPO_LOCAL_PATH = "/content/iss_group_24_src"


In [ ]:
from google.colab import drive


def mount_drive() -> str:
    """Mount Google Drive and return the iss_group_24 project root path."""
    drive.mount("/content/drive")
    for candidate in [
        "/content/drive/Shareddrives/iss_group_24",
        "/content/drive/MyDrive/iss_group_24",
    ]:
        if Path(candidate).exists():
            print(f"Drive mounted. Project root: {candidate}")
            return candidate
    raise RuntimeError(
        "Shared folder 'iss_group_24' not found on this Drive.\n"
        f"Open the link and add a shortcut to your Drive: {SHARED_FOLDER_LINK}"
    )

DRIVE_ROOT = mount_drive()

In [3]:
DRIVE_ROOT = "."

In [4]:
def setup_repo() -> None:
    """Clone the repo at depth=1, or reset to origin/main if already cloned."""
    if Path(REPO_LOCAL_PATH).exists():
        subprocess.run(["git", "-C", REPO_LOCAL_PATH, "fetch", "origin"], check=True)
        subprocess.run(
            ["git", "-C", REPO_LOCAL_PATH, "reset", "--hard", "origin/main"],
            check=True,
        )
        print(f"Repo updated to origin/main: {REPO_LOCAL_PATH}")
    else:
        subprocess.run(
            ["git", "clone", "--depth=1", REPO_URL, REPO_LOCAL_PATH],
            check=True,
        )
        print(f"Repo cloned: {REPO_LOCAL_PATH}")
    sys.path.insert(0, REPO_LOCAL_PATH)


def install_deps() -> None:
    """Install packages not pre-installed on Colab (torch/torchvision are pre-installed)."""
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "-q",
            "pillow>=10.0", "scipy", "matplotlib>=3.7",
        ],
        check=True,
    )
    print("Dependencies ready")


In [5]:
MANIFEST  = f"{DRIVE_ROOT}/dataset/cleaned/manifest.json"
DATA_ROOT = f"{DRIVE_ROOT}/dataset/cleaned"
MODEL_DIR = f"{DRIVE_ROOT}/model"

# Set CWD to the Drive project root so relative paths written by
# train.py (analysis/, manifest parent, etc.) land on the Drive.
os.chdir(DRIVE_ROOT)
print(f"Working directory: {os.getcwd()}")

Working directory: /mnt/Onetera/iss_group_24


In [ ]:
setup_repo()

---
## Step 0 — Build Dataset Manifest (80/20 train/test)

Copies and indexes images into `dataset/cleaned/`. Manifest holds only `train` and `test` splits; the trainer derives K-fold CV from the train pool.
**Run once.** Idempotent: re-runs only when staged files are missing.


In [6]:
import aggregator
from pathlib import Path

train_dir = Path(DATA_ROOT) / 'train'
test_dir  = Path(DATA_ROOT) / 'test'
staged_ok = (
    Path(MANIFEST).exists()
    and train_dir.is_dir() and any(train_dir.iterdir())
    and test_dir.is_dir()  and any(test_dir.iterdir())
)
if staged_ok:
    print(f'Dataset already staged at {DATA_ROOT} — skipping aggregator.')
else:
    print('Running aggregator…')
    aggregator.main()


Dataset already staged at ./dataset/cleaned — skipping aggregator.


---
## Imports & helpers

`resume_from(name_or_path)` returns the value to pass to a `train_stage*` `resume` kwarg. Recognised filenames (resolved against `MODEL_DIR`):
- `last.pt`, `best.pt`
- `stage1_complete.pt`, `stage2_complete.pt`, `stage3_complete.pt`
- `ckpt_s1_epoch{E:03d}.pt`
- `ckpt_s2_epoch{E:03d}.pt`
- `ckpt_s3_epoch{E:03d}_fold{F}.pt`

Absolute paths pass through.


In [7]:
from modeling.train import train_stage1, train_stage2, train_stage3
from modeling.evaluate import run as evaluate_run
from modeling.plot import plot_all_from_jsons

ANALYSIS_DIR = f'{DRIVE_ROOT}/analysis'

def resume_from(name_or_path: str | None):
    """None → True (auto from last.pt). filename → MODEL_DIR/filename. abs path → as-is."""
    if name_or_path is None:
        return True
    if Path(name_or_path).is_absolute():
        return str(name_or_path)
    return f'{MODEL_DIR}/{name_or_path}'


---
## Shared kwargs

Every hyperparameter is here. Each per-stage cell pulls from this dict; override with cell-local kwargs.


In [8]:
SHARED_KWARGS = dict(
    manifest         = MANIFEST,
    data_root        = DATA_ROOT,
    out_dir          = MODEL_DIR,
    analysis_dir     = ANALYSIS_DIR,

    # CV (Stage 3 only)
    folds            = 3,
    fold_seed        = 42,

    # Episodes / batches
    episodes_per_epoch_s1 = 1500,
    episodes_per_epoch_s2 = 1500,
    episodes_per_epoch_s3 = 2000,
    val_episodes_s1  = 400,
    val_episodes_s2  = 400,
    val_episodes_s3  = 240,
    batch_size       = 16,
    num_workers      = 2,
    n_support        = 4,

    # LR per stage
    lr_heads_s1            = 3e-4,
    lr_heads_s2            = 2e-4,
    lr_backbone_upper_s2   = 5e-5,
    lr_heads_s3            = 1e-4,
    lr_backbone_upper_s3   = 5e-6,
    lr_backbone_lower_s3   = 5e-6,
    weight_decay           = 1e-4,
    grad_clip              = 1.0,
    warmup_frac            = 0.05,

    # Loss / regularisation
    presence_weight     = 1.0,
    attn_loss_weight    = 0.5,
    aux_weight          = 0.5,
    entropy_reg_weight  = 0.01,
    reg_l2_prior_weight = 1e-3,
    proto_norm_weight   = 1e-3,
    contrastive         = True,
    contrastive_weight  = 0.1,
    contrastive_temp    = 0.1,
    vicreg              = True,
    vicreg_weight       = 0.05,
    barlow              = True,
    barlow_weight       = 0.005,
    triplet             = False,
    triplet_weight      = 0.1,

    # Curriculum
    neg_prob_s1         = 0.40,
    neg_prob_s2         = 0.45,
    neg_prob_s3         = 0.50,
    hard_neg_ratio_s1   = 0.0,
    hard_neg_ratio_s2   = 0.30,
    hard_neg_ratio_s3   = 0.50,

    # Source-balanced sampler (sums to batch_size)
    source_mix          = {'vizwiz_base': 4, 'vizwiz_novel': 2, 'hots': 5, 'insdet': 5},

    # Augmentation
    augment             = True,
    augment_strength    = 1.0,
    multi_scale_s2      = True,
    multi_scale_s3      = True,
    multi_scale_sizes   = (192, 224, 256),

    # Validation TTA
    val_use_tta         = True,
    val_tta_sizes       = (224, 288),

    # Checkpoints
    save_stage_completion = True,
    keep_last_n           = 6,

    seed = 42,
)

# Knobs only the post-stage `evaluate_run` cells use.
EVAL_KWARGS = dict(
    manifest    = MANIFEST,
    data_root   = DATA_ROOT,
    split       = 'test',
    episodes    = 600,
    batch_size  = 8,
    num_workers = 2,
    neg_prob    = 0.5,
    use_tta     = True,
)


---
## Stage 1 — Head warmup (no CV)

**Goal**: Warm up the matcher heads (SupportTokenizer / SupportFusion / cross-attention / detection / presence) on every source. Backbone stays frozen.

- **Trainable**: heads only.
- **LR**: heads `3e-4`.
- **Validation**: sample of the manifest's `test` split (read-only probe).
- **Resume**: auto from `last.pt` if present (set `resume=False` to start fresh).
- **Per-epoch logging** dumps every loss + every metric (overall + per-source) to stdout.
- **Checkpoints**: `ckpt_s1_epoch{E:03d}.pt` per epoch, `stage1_complete.pt` at end.


In [ ]:
# Train Stage 1
train_stage1(
    **SHARED_KWARGS,
    stage1_epochs = 5,
    resume        = True,   # or resume_from('ckpt_s1_epoch003.pt') for mid-stage resume
)



=== Stage 1 (head warmup, 5 epochs) ===
┌─ s1 epoch 1/5  (wall=301.4s)
│  lr      : g0=2.82e-04
│  train   : loss=7.9969  qfl=0.6166  neg_qfl=0.0001  centerness=1.2244  dfl=1.2257  giou=2.7082  presence=0.1076
│  train   : attn=0.2405  aux=2.9408  entropy_reg=0.3977  reg_l2_prior=314.0779  proto_norm=8.5614  nt_xent=1.6699
│  train   : vicreg=0.5864  barlow=0.1512  triplet=0.0000  grad_norm=8.8839  n_steps=93
│  val     : n=400  n_pos=189  n_neg=211  map_50=0.0125  map_5095=0.0025  f1_50=0.0251  precision_50=0.0308  recall_50=0.0212
│  val     : iou_mean=0.0631  iou_median=0.0000  iou_p25=0.0000  iou_p75=0.0000  contain_mean=0.1128
│  val     : contain_at_iou_50=0.0212  contain_at_iou_75=0.0000  presence_acc=0.6500  presence_acc_pos=0.9259
│  val     : presence_acc_neg=0.4028  mean_score_pos=0.5879  mean_score_neg=0.5093  presence_auroc=0.7753
│  val     : presence_pr_auc=0.7320  presence_brier=0.2228  frac_pred_near_corner=0.0025  frac_pred_tiny_box=0.2050
│  val     : argmax_cell_en

### Evaluate Stage 1 on the test split

Full kitchen-sink eval (mAP / IoU / containment / presence / per-source / collapse diagnostics) against `dataset/cleaned/test`, using `stage1_complete.pt`. Report at `analysis/stage1/test_eval/test_report.json`.


In [ ]:
evaluate_run(
    **EVAL_KWARGS,
    checkpoint = f'{MODEL_DIR}/stage1_complete.pt',
    report     = f'{ANALYSIS_DIR}/stage1/test_eval/test_report.json',
)


---
## Stage 2 — Partial unfreeze (no CV)

**Goal**: Adapt backbone features to the few-shot episode distribution by partially unfreezing the upper MobileNetV3 blocks.

- **Trainable**: heads + `backbone.features[7:]`.
- **LR**: heads `2e-4`, backbone upper `5e-5`.
- **Curriculum**: hard-neg ratio 0.30; `neg_prob` 0.45.
- **Multi-scale training**: input size sampled per batch from {192, 224, 256}.
- **Validation**: sample of the manifest's `test` split.
- **Resume**: defaults to `stage1_complete.pt`.
- **Checkpoints**: `ckpt_s2_epoch{E:03d}.pt` per epoch, `stage2_complete.pt` at end.


In [ ]:
# Train Stage 2
train_stage2(
    **SHARED_KWARGS,
    stage2_epochs = 8,
    resume        = resume_from('stage1_complete.pt'),
)


### Evaluate Stage 2 on the test split

Same kitchen-sink eval, using `stage2_complete.pt`. Report at `analysis/stage2/test_eval/test_report.json`.


In [ ]:
evaluate_run(
    **EVAL_KWARGS,
    checkpoint = f'{MODEL_DIR}/stage2_complete.pt',
    report     = f'{ANALYSIS_DIR}/stage2/test_eval/test_report.json',
)


---
## Stage 3 — Full unfreeze + K-fold rotating CV

**Goal**: End-to-end fine-tuning with cross-fold validation as the main signal. Inside each Stage 3 epoch, the same model rotates through `K=3` folds: train fold-0 → val fold-0 → train fold-1 → val fold-1 → train fold-2 → val fold-2. Per-(epoch, fold) JSON at `analysis/stage3/epoch_NNN/fold_F.json`; per-epoch cross-fold mean / min / max / std at `aggregate.json`.

- **Trainable**: full backbone + heads.
- **LR**: heads `1e-4`, backbone upper `5e-6`, backbone lower `5e-6`.
- **Curriculum**: hard-neg ratio 0.50; `neg_prob` 0.50.
- **Multi-scale training + 2-scale validation TTA** active.
- **Resume**: defaults to `stage2_complete.pt`. Mid-stage resume from `ckpt_s3_epoch{E}_fold{F}.pt` continues at the next fold.
- **Checkpoints**: `ckpt_s3_epoch{E:03d}_fold{F}.pt` per (epoch, fold), `stage3_complete.pt` at end.


In [ ]:
# Train Stage 3
train_stage3(
    **SHARED_KWARGS,
    stage3_epochs = 35,
    folds         = 3,
    resume        = resume_from('stage2_complete.pt'),
)


### Evaluate Stage 3 on the test split

Final post-training evaluation on the held-out test split using `stage3_complete.pt`. Report at `analysis/stage3/test_eval/test_report.json`.

Use `f'{MODEL_DIR}/best.pt'` instead if you want the best-by-metric checkpoint across all stages.


In [ ]:
evaluate_run(
    **EVAL_KWARGS,
    checkpoint = f'{MODEL_DIR}/stage3_complete.pt',
    report     = f'{ANALYSIS_DIR}/stage3/test_eval/test_report.json',
)


---
## Generate the offline report

Reads every per-epoch / per-fold / aggregate JSON under `ANALYSIS_DIR` (plus the per-stage `test_eval/test_report.json`) and writes PNGs to `ANALYSIS_DIR/plots/`.


In [ ]:
plot_all_from_jsons(ANALYSIS_DIR)
